In [ ]:
import re
from IPython.display import display
import joblib
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import pandas as pd
from PyALE import ale
import seaborn as sns
import shap
from xgboost import XGBRegressor

In [ ]:
print("ANALIZA PROGÓW PSYCHOLOGICZNYCH ZA POMOCĄ WYKRESÓW ALE")

sns.set_theme(style="ticks")

KOLUMNA_PRZEBIEG = 'mileage'
KOLUMNA_WIEK = 'car_age'

# wczytywanie danych i modeli z plików
try:
    X_test_with_clus = pd.read_pickle('X_test_with_clusters.pkl')
    model_features = joblib.load('model_feature_names.pkl')
    model_rf = joblib.load('model_rf_optimized.pkl')
    
    model_xgb = XGBRegressor()
    model_xgb.load_model('model_xgb_optimized.json')
    print("Dane oraz oba zoptymalizowane modele wczytane pomyślnie.")
except FileNotFoundError as e:
    print(f"Nie znaleziono potrzebnych plików na dysku! {e}")
    raise

# przygotowanie danych z pełnym zbiorem testowym do obliczenia profili ALE
X_test_model_input = X_test_with_clus[model_features].copy()

def format_duzych_liczb(x, pos):
    return "{:,}".format(int(x)).replace(',', ' ')

kolor_xgb = '#1f77b4' 
kolor_rf = '#ff7f0e'  

# wykresy ALE dla przebiegu 
ale_eff_mil_xgb = ale(X=X_test_model_input, model=model_xgb, feature=[KOLUMNA_PRZEBIEG], grid_size=40, plot=False)
ale_eff_mil_rf = ale(X=X_test_model_input, model=model_rf, feature=[KOLUMNA_PRZEBIEG], grid_size=40, plot=False)

fig1, ax1 = plt.subplots(figsize=(8, 6))

ax1.plot(ale_eff_mil_xgb.index, ale_eff_mil_xgb['eff'], color=kolor_xgb, lw=2.5, label='XGBoost')
ax1.plot(ale_eff_mil_rf.index, ale_eff_mil_rf['eff'], color=kolor_rf, lw=2.5, linestyle='--', label='Random Forest')

ax1.set_xlabel('Przebieg pojazdu (km)', fontsize=11)
ax1.set_ylabel('Efekt lokalny na cenę ofertową (PLN)', fontsize=11)
ax1.xaxis.set_major_formatter(ticker.FuncFormatter(format_duzych_liczb))
ax1.yaxis.set_major_formatter(ticker.FuncFormatter(format_duzych_liczb))
ax1.grid(True, linestyle='--', alpha=0.5)
ax1.legend(frameon=True, loc='best')
sns.despine(ax=ax1)

plt.tight_layout()
plt.show()

# wykresy ALE dla wieku pojazdu
ale_eff_age_xgb = ale(X=X_test_model_input, model=model_xgb, feature=[KOLUMNA_WIEK], grid_size=20, plot=False)
ale_eff_age_rf = ale(X=X_test_model_input, model=model_rf, feature=[KOLUMNA_WIEK], grid_size=20, plot=False)

fig2, ax2 = plt.subplots(figsize=(8, 6))

ax2.plot(ale_eff_age_xgb.index, ale_eff_age_xgb['eff'], color=kolor_xgb, lw=2.5, label='XGBoost')
ax2.plot(ale_eff_age_rf.index, ale_eff_age_rf['eff'], color=kolor_rf, lw=2.5, linestyle='--', label='Random Forest')

ax2.set_xlabel('Wiek pojazdu (lata)', fontsize=11)
ax2.set_ylabel('Efekt lokalny na cenę ofertową (PLN)', fontsize=11)
ax2.yaxis.set_major_formatter(ticker.FuncFormatter(format_duzych_liczb))
ax2.grid(True, linestyle='--', alpha=0.5)
ax2.legend(frameon=True, loc='best')
sns.despine(ax=ax2)

plt.tight_layout()
plt.show()

In [ ]:
print(" OBLICZANIE I ZAPIS WARTOŚCI SHAP DLA XGBOOST")

# wczytywanie danych z klastrami i modelu XGBoost z dysku
try:
    X_test_with_clus = pd.read_pickle('X_test_with_clusters.pkl')
    model_features = joblib.load('model_feature_names.pkl')
    
    model_xgb = XGBRegressor()
    model_xgb.load_model('model_xgb_optimized.json')
    print("Pomyślnie wczytano dane z klastrami i model z dysku.")
except FileNotFoundError as e:
    print(f"Nie znaleziono potrzebnych plików na dysku {e}")
    raise


# przygotowanie pełnego zbioru danych
X_full_shap = X_test_with_clus[model_features].copy()

klastry_full_shap = X_test_with_clus['cluster_id'].copy()

# Wyliczenie wartości SHAP dla XGBoosta
explainer_xgb = shap.TreeExplainer(model_xgb)
shap_values_xgb = explainer_xgb(X_full_shap)

# zapis obiektów na dysk 
joblib.dump(shap_values_xgb, 'shap_values_xgb.pkl')
joblib.dump(X_full_shap, 'X_full_xai.pkl')
joblib.dump(klastry_full_shap, 'klastry_full_xai.pkl')


print("'shap_values_xgb.pkl' - obiekt z wartościami SHAP (pełna populacja)")
print("'X_full_xai.pkl' - pełny zbiór danych wejściowych")
print("'klastry_full_xai.pkl' - pełny wektor klastrów do analizy segmentowej")

In [ ]:
# wczytywanie obliczonych wartości SHAP i danych z dysku
try:
    shap_values_xgb = joblib.load('shap_values_xgb.pkl')
    X_full_shap = joblib.load('X_full_xai.pkl')
    klastry_full_shap = joblib.load('klastry_full_xai.pkl')
    print("Pomyślnie wczytano obliczone wartości SHAP oraz przypisania do klastrów.")
except FileNotFoundError as e:
    print(f"Błąd wczytywania: {e}")
    raise

In [ ]:
# wykresy globalnej ważności cech (SHAP) dla całego rynku
sns.set_theme(style="ticks")

tlumaczenia_shap = {
    'car_age': 'Wiek pojazdu',
    'vehicle_model': 'Model pojazdu',
    'power': 'Moc silnika',
    'mileage': 'Przebieg',
    'vehicle_brand': 'Marka pojazdu',
    'damaged': 'Uszkodzony',
    'engine_displacement': 'Pojemność skokowa',
    'drive_type_Front wheels': 'Napęd na przednie koła',
    'no_accident': 'Bezwypadkowy',
    'transmission_Automatic': 'Automatyczna skrzynia biegów',
    'transmission_Manual': 'Manualna skrzynia biegów',
    'registered_in_poland': 'Zarejestrowany w Polsce',
    'first_owner_from_new': 'Pierwszy właściciel',
    'serviced_at_aso': 'Serwisowany w ASO',
    'dealership_warranty_included': 'Gwarancja dealerska',
    'drive_type_Rear wheels': 'Napęd na tylne koła',
    'drive_type_4x4 (attached automatically)': 'Napęd 4x4 (dołączany automatycznie)',
    'drive_type_4x4 (fixed)': 'Napęd 4x4 (stały)',
    'drive_type_4x4 (manually attached)': 'Napęd 4x4 (dołączany ręcznie)',
    'fuel_type_Gasoline': 'Paliwo: Benzyna',
    'fuel_type_Diesel': 'Paliwo: Diesel',
    'fuel_type_Gasoline + LPG': 'Paliwo: Benzyna + LPG',
    'fuel_type_Hybrid': 'Paliwo: Hybryda',
    'fuel_type_Electric': 'Paliwo: Elektryczny',
    'body_type_SUV': 'Nadwozie: SUV',
    'body_type_Station wagon': 'Nadwozie: Kombi',
    'body_type_Compact': 'Nadwozie: Kompakt',
    'body_type_Sedan': 'Nadwozie: Sedan',
    'body_type_Minivan': 'Nadwozie: Minivan',
    'body_type_City cars': 'Nadwozie: Auta miejskie',
    'body_type_Coupe': 'Nadwozie: Coupe',
    'body_type_Small cars': 'Nadwozie: Małe auta',
    'body_type_Cabriolet': 'Nadwozie: Kabriolet',
    'seller_type_Private person': 'Sprzedawca: Osoba prywatna',
    'seller_type_Dealer': 'Sprzedawca: Dealer',
    'seller_type_Authorized Dealer': 'Sprzedawca: Autoryzowany Dealer',
    'from_poland': 'Kraj pochodzenia: Polska',
    'from_germany': 'Kraj pochodzenia: Niemcy',
    'from_france': 'Kraj pochodzenia: Francja',
    'from_usa': 'Kraj pochodzenia: USA',
    'from_other_abroad': 'Kraj pochodzenia: Inny',
    'from_origin_unknown': 'Nieznane pochodzenie'
}

def format_duzych_liczb(x, pos):
    return "{:,}".format(int(x)).replace(',', ' ')

def spolszcz_etykiety_shap(ax):
    labels = [item.get_text() for item in ax.get_yticklabels()]
    new_labels = []
    for label in labels:
        # obsługa agregacji pozostałych cech
        if "Sum of" in label and "other features" in label:
            match = re.search(r'\d+', label)
            if match:
                liczba = match.group()
                new_labels.append(f"Pozostałe ({liczba} cech)")
            else:
                new_labels.append("Pozostałe cechy")
        else:
            new_labels.append(tlumaczenia_shap.get(label, label))
    
    ax.set_yticklabels(new_labels)

# wykres słupkowy (Globalna średnia bezwzględna ważność cech)
plt.figure(figsize=(10, 6))

shap.plots.bar(shap_values_xgb, max_display=11, show=False)

plt.xlabel("Średnia bezwzględna wartość wpływu na cenę (PLN)", fontsize=11)

ax = plt.gca()
ax.xaxis.set_major_formatter(ticker.FuncFormatter(format_duzych_liczb))
spolszcz_etykiety_shap(ax)

plt.tight_layout()
plt.show()

# wykres roju Beeswarm - kierunek i rozkład wpływu
plt.figure(figsize=(10, 6))

shap.plots.beeswarm(shap_values_xgb, max_display=11, show=False)

plt.xlabel("Wartość SHAP (Wpływ na cenę ofertową w PLN)", fontsize=11)

ax = plt.gca()
ax.xaxis.set_major_formatter(ticker.FuncFormatter(format_duzych_liczb))
plt.xticks(rotation=45, ha='right')
spolszcz_etykiety_shap(ax)

cb_ax = plt.gcf().axes[-1] 
cb_ax.set_ylabel("Wartość cechy", fontsize=11)
cb_ax.set_yticklabels(['Niska', 'Wysoka'])

plt.tight_layout()
plt.show()

In [ ]:
# wizualizacja wpływu cech w podziale na klastry
unikalne_klastry = np.sort(np.unique(klastry_full_shap))

moderacja_dane = []

for cid in unikalne_klastry:
    maska_klastra = (klastry_full_shap == cid).values
    shap_values_klaster = shap_values_xgb[maska_klastra]

    print(f"\nWIZUALIZACJA DLA KLASTRA {cid}")

    # wykres typu Beeswarm specyficzny dla segmentu
    plt.figure(figsize=(10, 6))
    
    shap.plots.beeswarm(shap_values_klaster, max_display=11, show=False)

    plt.xlabel("Wartość SHAP (Wpływ na cenę ofertową w PLN)", fontsize=11)

    ax = plt.gca()
    ax.xaxis.set_major_formatter(ticker.FuncFormatter(format_duzych_liczb))
    plt.xticks(rotation=45, ha='right')
    
    spolszcz_etykiety_shap(ax)

    fig = plt.gcf()
    if len(fig.axes) > 1: 
        cb_ax = fig.axes[-1]
        cb_ax.set_ylabel("Wartość cechy", fontsize=11)
        cb_ax.set_yticklabels(['Niska', 'Wysoka'])

    plt.tight_layout()
    plt.show()

    # zebranie średnich bezwzględnych wartości SHAP dla kluczowych cech
    idx_age = list(X_full_shap.columns).index('car_age')
    idx_mileage = list(X_full_shap.columns).index('mileage')

    moderacja_dane.append({
        'Klaster / Segment': f'Klaster {cid}',
        'Średni wpływ Wieku (PLN)': round(mean_abs_shap[idx_age], 2),
        'Średni wpływ Przebiegu (PLN)': round(mean_abs_shap[idx_mileage], 2),
        'Liczba aut w analizie': maska_klastra.sum()
    })

# tabela zbiorcza
df_moderacja = pd.DataFrame(moderacja_dane)
display(df_moderacja)

In [ ]:
# tabela Top 10 zmiennych w zależności od klastra (Ujęcie procentowe)

# obliczenie globalnego wpływu oraz udziałów procentowych
global_mean_abs_shap = np.abs(shap_values_xgb.values).mean(axis=0)
global_total_shap = global_mean_abs_shap.sum() 
global_pct_shap = (global_mean_abs_shap / global_total_shap) * 100

feature_names = list(X_full_shap.columns)

df_global = pd.DataFrame({
    'Nazwa_Oryginalna': feature_names,
    'Wplyw_Globalny_Pct': global_pct_shap
})

# wybór 10 najbardziej wpływowych cech na poziomie globalnym
top_10_features = df_global.sort_values(by='Wplyw_Globalny_Pct', ascending=False).head(10)
top_10_oryginalne = top_10_features['Nazwa_Oryginalna'].tolist()

tabela_top10 = {
    'Cecha': [tlumaczenia_shap.get(f, f) for f in top_10_oryginalne],
    'Wpływ Globalny (%)': top_10_features['Wplyw_Globalny_Pct'].values
}

# obliczenia rozbicia na poszczególne klastry
unikalne_klastry = np.sort(np.unique(klastry_full_shap))

for cid in unikalne_klastry:
    maska_klastra = (klastry_full_shap == cid).values
    shap_klaster_values = shap_values_xgb.values[maska_klastra]
    
    # Obliczenie udziałów procentowych wewnątrz danego klastra
    klaster_mean_abs_shap = np.abs(shap_klaster_values).mean(axis=0)
    klaster_total_shap = klaster_mean_abs_shap.sum()
    klaster_pct_shap = (klaster_mean_abs_shap / klaster_total_shap) * 100
    
    # Wyciągnięcie procentowych wartości tylko dla 10 globalnych cech
    wartosci_dla_klastra = []
    for feat in top_10_oryginalne:
        idx = feature_names.index(feat)
        wartosci_dla_klastra.append(klaster_pct_shap[idx])
        
    tabela_top10[f'Klaster {cid} (%)'] = wartosci_dla_klastra

# utworzenie DataFrame z tabeli top 10 zmiennych
df_top10_final = pd.DataFrame(tabela_top10)

format_dict = {'Wpływ Globalny (%)': "{:.1f}%".format}
for cid in unikalne_klastry:
    format_dict[f'Klaster {cid} (%)'] = "{:.1f}%".format

print("TOP 10 ZMIENNYCH: UDZIAŁ PROCENTOWY W DECYZJACH MODELU")
display(df_top10_final.style.format(format_dict).apply(
    lambda x: ['background: #f0f0f0' if i%2==0 else '' for i in range(len(x))], axis=0
))